<a href="https://colab.research.google.com/github/Yisihaq01/breast-histology-calibration/blob/main/Notebooks/04_external_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
import shutil

EXTERNAL_ROOT = Path("/content/external_data")

BACH_ROOT = EXTERNAL_ROOT / "BACH"
BRACS_ROOT = EXTERNAL_ROOT / "BRACS"

for folder in [
    BACH_ROOT / "archives",
    BACH_ROOT / "images",
    BRACS_ROOT / "archives",
    BRACS_ROOT / "images",
    EXTERNAL_ROOT / "metadata",
]:
    folder.mkdir(parents=True, exist_ok=True)

disk = shutil.disk_usage("/content")

print("External data folder:", EXTERNAL_ROOT)
print(f"Free storage: {disk.free / 1024**3:.1f} GB")
print("\nBACH and BRACS folders are ready.")
print("Files in /content are temporary and need saving before runtime deletion.")

External data folder: /content/external_data
Free storage: 87.2 GB

BACH and BRACS folders are ready.
Files in /content are temporary and need saving before runtime deletion.


In [ ]:
import hashlib
import requests
from tqdm.auto import tqdm

BACH_URL = (
    "https://zenodo.org/records/3632035/files/"
    "ICIAR2018_BACH_Challenge.zip?download=1"
)

BACH_ARCHIVE = (
    BACH_ROOT / "archives" / "ICIAR2018_BACH_Challenge.zip"
)

EXPECTED_MD5 = "8ae1801334aa943c44627c1eef3631b2"


def file_md5(path):
    digest = hashlib.md5()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(8 * 1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


if BACH_ARCHIVE.exists():
    print("Archive already exists. Checking it before reuse.")

else:
    partial_path = BACH_ARCHIVE.with_suffix(".zip.part")

    with requests.get(
        BACH_URL,
        stream=True,
        timeout=(30, 180),
    ) as response:

        response.raise_for_status()
        total = int(response.headers.get("Content-Length", 0))

        with partial_path.open("wb") as file, tqdm(
            total=total or None,
            unit="B",
            unit_scale=True,
            desc="Downloading BACH",
        ) as progress:

            for chunk in response.iter_content(
                chunk_size=4 * 1024 * 1024
            ):
                if chunk:
                    file.write(chunk)
                    progress.update(len(chunk))

    actual_md5 = file_md5(partial_path)

    if actual_md5 != EXPECTED_MD5:
        raise RuntimeError(
            f"Checksum mismatch: {actual_md5}. "
            "The partial archive must not be used."
        )

    partial_path.rename(BACH_ARCHIVE)


actual_md5 = file_md5(BACH_ARCHIVE)

assert actual_md5 == EXPECTED_MD5, (
    f"Archive checksum mismatch: {actual_md5}"
)

print("\nPASS: BACH archive checksum verified.")
print("Archive:", BACH_ARCHIVE)
print(f"Size: {BACH_ARCHIVE.stat().st_size / 1024**3:.2f} GiB")


PASS: BACH archive checksum verified.
Archive: /content/external_data/BACH/archives/ICIAR2018_BACH_Challenge.zip
Size: 9.70 GiB


In [ ]:
from collections import Counter
from pathlib import PurePosixPath
import zipfile

assert BACH_ARCHIVE.is_file()

with zipfile.ZipFile(BACH_ARCHIVE, "r") as archive:
    archive_files = [
        entry
        for entry in archive.infolist()
        if not entry.is_dir()
    ]

    print("Files in archive:", len(archive_files))

    folder_counts = Counter()
    image_counts = Counter()
    metadata_files = []

    image_extensions = {".tif", ".tiff", ".png", ".jpg", ".jpeg"}
    metadata_extensions = {
        ".csv", ".xlsx", ".xls", ".txt", ".md", ".json"
    }

    for entry in archive_files:
        path = PurePosixPath(entry.filename)

        # Ignore macOS bookkeeping files.
        if "__MACOSX" in path.parts or path.name.startswith("._"):
            continue

        folder = "/".join(path.parts[:2])
        folder_counts[folder] += 1

        if path.suffix.lower() in image_extensions:
            image_counts[str(path.parent)] += 1

        if path.suffix.lower() in metadata_extensions:
            metadata_files.append(entry.filename)

    print("\nTop-level folder counts:")
    for folder, count in sorted(folder_counts.items()):
        print(f"{folder}: {count}")

    print("\nImage counts by folder:")
    for folder, count in sorted(image_counts.items()):
        print(f"{folder}: {count}")

    print("\nPotential metadata/documentation files:")
    if metadata_files:
        for filename in sorted(metadata_files):
            print(filename)
    else:
        print("None found.")

    print("\nFirst 10 image paths:")
    image_paths = [
        entry.filename
        for entry in archive_files
        if PurePosixPath(entry.filename).suffix.lower()
        in image_extensions
        and "__MACOSX" not in PurePosixPath(entry.filename).parts
        and not PurePosixPath(entry.filename).name.startswith("._")
    ]

    for filename in sorted(image_paths)[:10]:
        print(filename)

print("\nPASS: Archive inventory completed. No files extracted.")

Files in archive: 488

Top-level folder counts:
ICIAR2018_BACH_Challenge/Photos: 407
ICIAR2018_BACH_Challenge/WSI: 81

Image counts by folder:
ICIAR2018_BACH_Challenge/Photos/Benign: 100
ICIAR2018_BACH_Challenge/Photos/InSitu: 100
ICIAR2018_BACH_Challenge/Photos/Invasive: 100
ICIAR2018_BACH_Challenge/Photos/Normal: 100
ICIAR2018_BACH_Challenge/WSI/gt_thumbnails: 10
ICIAR2018_BACH_Challenge/WSI/thumbnails: 30

Potential metadata/documentation files:
ICIAR2018_BACH_Challenge/Photos/microscopy_ground_truth.csv

First 10 image paths:
ICIAR2018_BACH_Challenge/Photos/Benign/b001.tif
ICIAR2018_BACH_Challenge/Photos/Benign/b002.tif
ICIAR2018_BACH_Challenge/Photos/Benign/b003.tif
ICIAR2018_BACH_Challenge/Photos/Benign/b004.tif
ICIAR2018_BACH_Challenge/Photos/Benign/b005.tif
ICIAR2018_BACH_Challenge/Photos/Benign/b006.tif
ICIAR2018_BACH_Challenge/Photos/Benign/b007.tif
ICIAR2018_BACH_Challenge/Photos/Benign/b008.tif
ICIAR2018_BACH_Challenge/Photos/Benign/b009.tif
ICIAR2018_BACH_Challenge/Photos/

In [ ]:
from pathlib import PurePosixPath
import shutil
import zipfile
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

BACH_CLASSES = ("Normal", "Benign", "InSitu", "Invasive")
PHOTOS_PREFIX = ("ICIAR2018_BACH_Challenge", "Photos")

BACH_IMAGES = BACH_ROOT / "images"
BACH_METADATA = BACH_ROOT / "metadata"

BACH_METADATA.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(BACH_ARCHIVE, "r") as archive:
    selected_images = []

    for entry in archive.infolist():
        path = PurePosixPath(entry.filename)

        if (
            not entry.is_dir()
            and path.parts[:2] == PHOTOS_PREFIX
            and len(path.parts) == 4
            and path.parts[2] in BACH_CLASSES
            and path.suffix.lower() in {".tif", ".tiff"}
            and not path.name.startswith("._")
        ):
            selected_images.append(entry)

    assert len(selected_images) == 400
    assert len({entry.filename for entry in selected_images}) == 400

    for entry in tqdm(
        selected_images,
        desc="Extracting microscopy images",
    ):
        path = PurePosixPath(entry.filename)
        destination = BACH_IMAGES / path.parts[2] / path.name
        destination.parent.mkdir(parents=True, exist_ok=True)

        with archive.open(entry) as source:
            with destination.open("wb") as target:
                shutil.copyfileobj(source, target)

    ground_truth_member = (
        "ICIAR2018_BACH_Challenge/Photos/"
        "microscopy_ground_truth.csv"
    )

    ground_truth_path = (
        BACH_METADATA / "microscopy_ground_truth.csv"
    )

    # Preserve the original CSV exactly as supplied.
    with archive.open(ground_truth_member) as source:
        with ground_truth_path.open("wb") as target:
            shutil.copyfileobj(source, target)


print("\nExtracted microscopy images:")

for class_name in BACH_CLASSES:
    count = sum(
        path.suffix.lower() in {".tif", ".tiff"}
        for path in (BACH_IMAGES / class_name).iterdir()
        if path.is_file()
    )

    assert count == 100, (
        f"Unexpected count for {class_name}: {count}"
    )

    print(f"{class_name}: {count}")


# The original CSV has no header row.
ground_truth = pd.read_csv(
    ground_truth_path,
    header=None,
    names=["filename", "original_class"],
    dtype=str,
    keep_default_na=False,
)

for column in ground_truth.columns:
    ground_truth[column] = ground_truth[column].str.strip()

assert len(ground_truth) == 400
assert not ground_truth["filename"].duplicated().any()
assert set(ground_truth["original_class"]) == set(BACH_CLASSES)

# Save a separate copy with descriptive headers.
headered_ground_truth_path = (
    BACH_METADATA / "microscopy_ground_truth_with_headers.csv"
)

ground_truth.to_csv(
    headered_ground_truth_path,
    index=False,
)

print("\nGround-truth CSV with headers:")
print("Shape:", ground_truth.shape)
print("Columns:", ground_truth.columns.tolist())

display(ground_truth.head(10))

print("\nPASS: 400 microscopy images and labels loaded.")
print("Original CSV:", ground_truth_path)
print("CSV with headers:", headered_ground_truth_path)

Extracting microscopy images:   0%|          | 0/400 [00:00<?, ?it/s]


Extracted microscopy images:
Normal: 100
Benign: 100
InSitu: 100
Invasive: 100

Ground-truth CSV with headers:
Shape: (400, 2)
Columns: ['filename', 'original_class']


,filename,original_class
0,n001.tif,Normal
1,n002.tif,Normal
2,n003.tif,Normal
3,n004.tif,Normal
4,n005.tif,Normal
5,n006.tif,Normal
6,n007.tif,Normal
7,n008.tif,Normal
8,n009.tif,Normal
9,n010.tif,Normal



PASS: 400 microscopy images and labels loaded.
Original CSV: /content/external_data/BACH/metadata/microscopy_ground_truth.csv
CSV with headers: /content/external_data/BACH/metadata/microscopy_ground_truth_with_headers.csv


## Verify labels and define the binary evaluation target

We compare the ground-truth CSV against the extracted image
filenames and class folders before creating evaluation metadata.

Binary mapping:
- Normal and Benign: 0 (benign)
- InSitu and Invasive: 1 (malignant)

BACH is reserved for external evaluation. Its original challenge
training set serves as external evaluation data for our
BreakHis-trained model.

Patient identifiers remain unverified. We do not infer patient
identity from image filenames.

In [ ]:
# Build an independent inventory from the extracted image folders.
image_records = []

for class_name in BACH_CLASSES:
    for path in sorted((BACH_IMAGES / class_name).iterdir()):
        if (
            path.is_file()
            and path.suffix.lower() in {".tif", ".tiff"}
        ):
            image_records.append({
                "filename": path.name,
                "folder_class": class_name,
                "relative_path": (
                    path.relative_to(BACH_ROOT).as_posix()
                ),
            })

image_inventory = pd.DataFrame(image_records)

assert len(image_inventory) == 400
assert not image_inventory["filename"].duplicated().any()

# Compare every CSV record against the extracted files.
verified_labels = ground_truth.merge(
    image_inventory,
    on="filename",
    how="outer",
    validate="one_to_one",
    indicator=True,
)

assert verified_labels["_merge"].eq("both").all(), (
    "Ground-truth records and extracted images do not match."
)

assert verified_labels["original_class"].eq(
    verified_labels["folder_class"]
).all(), "CSV labels disagree with image-folder labels."

# Prespecified mapping for external binary evaluation.
BACH_LABEL_MAP = {
    "Normal": 0,
    "Benign": 0,
    "InSitu": 1,
    "Invasive": 1,
}

bach_metadata = (
    verified_labels
    .drop(columns=["_merge", "folder_class"])
    .sort_values("filename")
    .reset_index(drop=True)
)

bach_metadata["label"] = (
    bach_metadata["original_class"]
    .map(BACH_LABEL_MAP)
    .astype(int)
)

bach_metadata["dataset"] = "BACH"
bach_metadata["source_subset"] = "challenge_training_microscopy"
bach_metadata["role"] = "external_evaluation_only"

# Leave patient identity unknown until supported by metadata.
bach_metadata["patient_id"] = pd.NA

print("Verified images:", len(bach_metadata))

print("\nOriginal class counts:")
print(
    bach_metadata["original_class"]
    .value_counts()
    .sort_index()
)

print("\nBinary counts (0 = benign, 1 = malignant):")
print(bach_metadata["label"].value_counts().sort_index())

assert bach_metadata["label"].value_counts().to_dict() == {
    0: 200,
    1: 200,
}

display(bach_metadata.head())

print("\nPASS: All CSV labels match the extracted image folders.")
print("Binary mapping verified: 200 benign, 200 malignant.")
print("Patient IDs remain unverified.")

Verified images: 400

Original class counts:
original_class
Benign      100
InSitu      100
Invasive    100
Normal      100
Name: count, dtype: int64

Binary counts (0 = benign, 1 = malignant):
label
0    200
1    200
Name: count, dtype: int64


,filename,original_class,relative_path,label,dataset,source_subset,role,patient_id
0,b001.tif,Benign,images/Benign/b001.tif,0,BACH,challenge_training_microscopy,external_evaluation_only,<NA>
1,b002.tif,Benign,images/Benign/b002.tif,0,BACH,challenge_training_microscopy,external_evaluation_only,<NA>
2,b003.tif,Benign,images/Benign/b003.tif,0,BACH,challenge_training_microscopy,external_evaluation_only,<NA>
3,b004.tif,Benign,images/Benign/b004.tif,0,BACH,challenge_training_microscopy,external_evaluation_only,<NA>
4,b005.tif,Benign,images/Benign/b005.tif,0,BACH,challenge_training_microscopy,external_evaluation_only,<NA>



PASS: All CSV labels match the extracted image folders.
Binary mapping verified: 200 benign, 200 malignant.
Patient IDs remain unverified.


## Audit image integrity, dimensions and duplicates

We decode all 400 microscopy images and record their dimensions,
color modes and frame counts.

We detect exact duplicates using SHA-256 hashes of decoded RGB
pixels, including image dimensions in the hash. We preserve all
original files and review duplicate groups before making any
exclusions.

This audit does not establish whether different images come
from the same patient.

In [ ]:
import hashlib
import struct

from PIL import Image
from tqdm.auto import tqdm

audit_records = []
audit_errors = []

for row in tqdm(
    bach_metadata.itertuples(index=False),
    total=len(bach_metadata),
    desc="Auditing BACH images",
):
    image_path = BACH_ROOT / row.relative_path

    try:
        # Check file integrity, then reopen for full decoding.
        with Image.open(image_path) as image:
            image.verify()

        with Image.open(image_path) as image:
            original_mode = image.mode
            image_format = image.format
            frame_count = getattr(image, "n_frames", 1)

            image.seek(0)
            rgb = image.convert("RGB")
            rgb.load()

            width, height = rgb.size

            digest = hashlib.sha256()
            digest.update(struct.pack(">II", width, height))
            digest.update(rgb.tobytes())

        audit_records.append({
            "filename": row.filename,
            "width": width,
            "height": height,
            "original_mode": original_mode,
            "image_format": image_format,
            "frame_count": frame_count,
            "file_bytes": image_path.stat().st_size,
            "rgb_sha256": digest.hexdigest(),
        })

    except Exception as error:
        audit_errors.append({
            "filename": row.filename,
            "error": f"{type(error).__name__}: {error}",
        })


bach_image_audit = pd.DataFrame(audit_records)

print("\nSuccessfully decoded:", len(bach_image_audit))
print("Failed images:", len(audit_errors))

if audit_errors:
    display(pd.DataFrame(audit_errors))

assert not audit_errors, (
    "Image audit failed. Review errors before continuing."
)
assert len(bach_image_audit) == 400
assert bach_image_audit["width"].gt(0).all()
assert bach_image_audit["height"].gt(0).all()

print("\nImage dimensions:")
display(
    bach_image_audit.groupby(["width", "height"])
    .size()
    .reset_index(name="images")
)

print("\nOriginal color modes:")
print(bach_image_audit["original_mode"].value_counts())

print("\nFrame counts:")
print(bach_image_audit["frame_count"].value_counts())

# Attach audit results without changing the original metadata.
bach_audited_metadata = bach_metadata.merge(
    bach_image_audit,
    on="filename",
    how="left",
    validate="one_to_one",
)

duplicate_mask = bach_audited_metadata.duplicated(
    "rgb_sha256",
    keep=False,
)

bach_duplicate_audit = (
    bach_audited_metadata.loc[
        duplicate_mask,
        [
            "rgb_sha256",
            "filename",
            "original_class",
            "label",
            "relative_path",
        ],
    ]
    .sort_values(["rgb_sha256", "filename"])
    .reset_index(drop=True)
)

duplicate_groups = bach_duplicate_audit["rgb_sha256"].nunique()

print("\nUnique decoded RGB images:",
      bach_image_audit["rgb_sha256"].nunique())
print("Exact duplicate groups:", duplicate_groups)
print("Files in duplicate groups:", len(bach_duplicate_audit))

if duplicate_groups:
    display(bach_duplicate_audit)

    print(
        "\nDuplicate groups with conflicting original classes:",
        (
            bach_duplicate_audit.groupby("rgb_sha256")
            ["original_class"].nunique().gt(1).sum()
        ),
    )

    print(
        "Duplicate groups with conflicting binary labels:",
        (
            bach_duplicate_audit.groupby("rgb_sha256")
            ["label"].nunique().gt(1).sum()
        ),
    )

print("\nPASS: Image audit completed. No files removed.")

Auditing BACH images:   0%|          | 0/400 [00:00<?, ?it/s]


Successfully decoded: 400
Failed images: 0

Image dimensions:


,width,height,images
0,2048,1536,400



Original color modes:
original_mode
RGB    400
Name: count, dtype: int64

Frame counts:
frame_count
1    400
Name: count, dtype: int64

Unique decoded RGB images: 400
Exact duplicate groups: 0
Files in duplicate groups: 0

PASS: Image audit completed. No files removed.


## Save the verified BACH metadata

All 400 microscopy images passed decoding and have dimensions
2048 × 1536 pixels. Each image contains one RGB frame.
The decoded-pixel audit found no exact duplicates.

We retain all 400 images for external evaluation and save the
label mapping and audit results. Patient identifiers remain
unverified. Physical-scale normalization and patch extraction
are pending.

In [ ]:
import json

BACH_METADATA.mkdir(parents=True, exist_ok=True)

metadata_path = BACH_METADATA / "bach_verified_metadata.csv"
image_audit_path = BACH_METADATA / "bach_image_audit.csv"
duplicate_audit_path = BACH_METADATA / "bach_duplicate_audit.csv"
label_map_path = BACH_METADATA / "bach_binary_label_mapping.json"
summary_path = BACH_METADATA / "bach_audit_summary.json"

# Save small, reproducible metadata files.
bach_audited_metadata.to_csv(metadata_path, index=False)
bach_image_audit.to_csv(image_audit_path, index=False)
bach_duplicate_audit.to_csv(duplicate_audit_path, index=False)

label_mapping = {
    "dataset": "BACH",
    "mapping": BACH_LABEL_MAP,
    "binary_labels": {
        "0": "benign",
        "1": "malignant",
    },
    "role": "external_evaluation_only",
    "note": (
        "Normal tissue is included in the benign binary category. "
        "Original four-class labels are preserved."
    ),
}

audit_summary = {
    "dataset": "BACH",
    "source": "https://zenodo.org/records/3632035",
    "source_subset": "challenge_training_microscopy",
    "archive_md5": EXPECTED_MD5,
    "n_images": int(len(bach_audited_metadata)),
    "class_counts": {
        str(name): int(count)
        for name, count in
        bach_audited_metadata["original_class"].value_counts().items()
    },
    "binary_counts": {
        str(label): int(count)
        for label, count in
        bach_audited_metadata["label"].value_counts().items()
    },
    "dimensions": [
        {
            "width": int(width),
            "height": int(height),
            "images": int(count),
        }
        for (width, height), count in
        bach_image_audit.groupby(["width", "height"]).size().items()
    ],
    "failed_images": int(len(audit_errors)),
    "exact_duplicate_groups": int(duplicate_groups),
    "patient_ids_verified": False,
    "preprocessing_status": "pending",
}

for path, content in [
    (label_map_path, label_mapping),
    (summary_path, audit_summary),
]:
    path.write_text(
        json.dumps(content, indent=2) + "\n",
        encoding="utf-8",
    )

# Reopen saved metadata to verify the export.
saved_metadata = pd.read_csv(
    metadata_path,
    dtype={"filename": str, "patient_id": str},
)

assert len(saved_metadata) == 400
assert saved_metadata["filename"].is_unique
assert saved_metadata["rgb_sha256"].nunique() == 400
assert saved_metadata["patient_id"].isna().all()
assert saved_metadata["label"].value_counts().to_dict() == {
    0: 200,
    1: 200,
}

saved_files = [
    metadata_path,
    image_audit_path,
    duplicate_audit_path,
    label_map_path,
    summary_path,
]

print("Saved audit files:")
for path in saved_files:
    print(f"- {path.name} ({path.stat().st_size:,} bytes)")

# Bundle only the audit files for downloading to your laptop.
audit_bundle = shutil.make_archive(
    str(BACH_ROOT / "bach_audit_outputs"),
    "zip",
    root_dir=BACH_METADATA,
    base_dir=".",
)

print("\nAudit bundle:", audit_bundle)
print("\nPASS: BACH metadata saved and verified.")

Saved audit files:
- bach_verified_metadata.csv (79,737 bytes)
- bach_image_audit.csv (41,864 bytes)
- bach_duplicate_audit.csv (55 bytes)
- bach_binary_label_mapping.json (329 bytes)
- bach_audit_summary.json (574 bytes)

Audit bundle: /content/external_data/BACH/bach_audit_outputs.zip

PASS: BACH metadata saved and verified.
